# Phase 40 — 2-honest-worker learning run (full rate + η-adaptation)

Follow-up to the single-worker run. **Two honest workers, one Qwen pinned per GPU.** This
fixes the two throttles of the single-worker run:

1. **Full rate** — `TARGET_PROPOSALS=2` is met every round by two real proposers, so the
   server advances ~once per iteration (vs every *two* iterations before): ~2× the gate updates.
2. **η-adaptation turns on** — with two workers the no-self-audit rule lets each worker close
   the *other's* pending audits, so sym-AIMD fires and η can climb above its 1e-3 init
   (single-worker runs never close an audit → η stayed frozen).

It also exercises the **≥2-honest-worker regime your paper §10 lists as untested.**

## Before ▶ Run All
1. Right sidebar (gear) → **Accelerator** → `GPU T4 x2` (two GPUs — important; falls back to
   sharing one GPU if only one is present, slower and tighter on memory).
2. **Internet** → `On`.  3. Run top to bottom.

~45–60 min on T4×2 for R=300. Resets the deployed coordinator first. Outputs in
`/kaggle/working/learn2/`: `traj-0.csv`, `traj-1.csv`, `descent2.png`, `RESULT2.md`.


## Cell 1 — install deps + clone repos (pulls the updated verifier)

In [ ]:
import subprocess, os, sys, urllib.request
try:
    urllib.request.urlopen("https://github.com", timeout=5).close()
    print("✓ internet reachable")
except Exception as e:
    sys.exit(f"✗ INTERNET DISABLED: {e}\n  Right sidebar (gear) → Internet → On, then re-run.")
print("→ pip install …")
subprocess.run(["pip", "-q", "install", "transformers", "torch", "numpy", "requests", "matplotlib"], check=True)
os.makedirs("/kaggle/working", exist_ok=True)
for name, url in [("ntkmirror", "https://github.com/leochlon/ntkmirror.git"),
                  ("postnet-cf", "https://github.com/abgnydn/postnet-cf.git")]:
    dst = f"/kaggle/working/{name}"
    if os.path.isdir(dst):
        print(f"→ git pull {name}"); subprocess.run(["git", "-C", dst, "pull", "--ff-only"], check=True)
    else:
        print(f"→ git clone {name}"); subprocess.run(["git", "clone", url, dst], check=True)
print("→ pip install -e ntkmirror")
subprocess.run(["pip", "-q", "install", "-e", "/kaggle/working/ntkmirror"], check=True)
print("OK")

## Cell 2 — build held-out corpus (32 train / 32 eval, disjoint)

Identical corpus to the single-worker run, so the two results are directly comparable.

In [ ]:
import json, random, os
os.makedirs("/kaggle/working/learn2", exist_ok=True)

def solve(a, b):
    o = (a % 10) + (b % 10)
    t = (a // 10) + (b // 10) + (o // 10)
    return (f" Add ones: {a%10}+{b%10}={o}, write {o%10} carry {o//10}. "
            f"Tens: {a//10}+{b//10}+{o//10}={t}. Answer: {a+b}")

rng = random.Random(40)
pairs = set()
while len(pairs) < 64:
    a, b = rng.randint(10, 89), rng.randint(10, 89)
    if a + b < 100:
        pairs.add((a, b))
pairs = list(pairs); rng.shuffle(pairs)
train, evl = pairs[:32], pairs[32:64]
assert not (set(train) & set(evl))

def write(path, ps):
    with open(path, "w") as f:
        for a, b in ps:
            f.write(json.dumps({"prompt": f"Problem: {a} + {b} = ?\nSolution:",
                                "completion": solve(a, b)}) + "\n")
write("/kaggle/working/learn2/train.jsonl", train)
write("/kaggle/working/learn2/eval.jsonl", evl)
print(f"train={len(train)} eval={len(evl)} disjoint ✓")

## Cell 3 — GPU check (expect 2) + reset coordinator

In [ ]:
import torch, requests
assert torch.cuda.is_available(), "No GPU. Right sidebar → Accelerator → GPU T4 x2."
N_GPU = torch.cuda.device_count()
print(f"✓ {N_GPU} GPU(s):", [torch.cuda.get_device_name(i) for i in range(N_GPU)])
if N_GPU < 2:
    print("⚠ only 1 GPU — both workers will SHARE it (slower, tighter memory). "
          "For the intended run pick 'GPU T4 x2'.")

COORD = "https://postnet-cf.abgunaydin94.workers.dev"
_UA = ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
       "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36 postnet-ntk/1.0")
_S = requests.Session(); _S.headers.update({"User-Agent": _UA})
print("reset:", _S.post(f"{COORD}/api/ntk/reset").json())
s = _S.get(f"{COORD}/api/ntk/state").json()
print(f"R={s['round']} eta={s['eta']} K={s.get('K')} target={s.get('target')}")

## Cell 4 — two honest workers, concurrent, one per GPU

Both train on the same batch and report `audit_loss_before` every round, so each closes the
other's pending audits → sym-AIMD adapts η. Neither passes `--reset` (Cell 3 already reset);
both bootstrap θ=0 from the freshly-reset server. Worker *i* is pinned to GPU `i % N_GPU`.

In [ ]:
import subprocess, time, os

ROUNDS = 300
def launch(idx, seed):
    gpu = idx % N_GPU
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    log = f"/kaggle/working/learn2/worker-{idx}.log"
    cmd = [
        "python", "/kaggle/working/postnet-cf/scripts/ntk-verifier.py",
        "--coord", COORD, "--model", "Qwen/Qwen2.5-0.5B-Instruct",
        "--train",    "/kaggle/working/learn2/train.jsonl",
        "--eval",     "/kaggle/working/learn2/eval.jsonl",
        "--artifact", "/kaggle/working/postnet-cf/public/data/qwen05b-math-gates-k5000.bin",
        "--rounds", str(ROUNDS), "--trials", "4",
        "--batch-size", "32", "--eval-batch-size", "32", "--max-length", "64",
        "--device", "cuda", "--dtype", "fp32",
        "--seed", str(seed), "--worker-id", f"kaggle-honest-{idx}",
        "--trajectory", f"/kaggle/working/learn2/traj-{idx}.csv",
    ]
    return subprocess.Popen(cmd, env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT), log

t0 = time.time()
(p0, l0) = launch(0, 1)
time.sleep(8)            # small stagger so both join before the first advance
(p1, l1) = launch(1, 2)
print("both workers launched; waiting…")
p0.wait(); p1.wait()
print(f"done in {time.time()-t0:.0f}s   exit: w0={p0.returncode} w1={p1.returncode}")
for l in (l0, l1):
    print(f"\n===== {l} (tail) =====")
    print("\n".join(open(l).read().splitlines()[-12:]))

## Cell 5 — plot + RESULT2.md (compared against the single-worker baseline)

In [ ]:
import csv, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def load(path):
    R, T, E = [], [], []
    with open(path) as f:
        for row in csv.DictReader(f):
            R.append(int(row["round"])); T.append(float(row["train_loss"]))
            E.append(float(row["eval_loss"]) if row["eval_loss"] else None)
    return R, T, E

# worker-0 tracks the shared global θ; use it as the canonical trajectory.
R0, T0, E0 = load("/kaggle/working/learn2/traj-0.csv")
eta_final = None
with open("/kaggle/working/learn2/traj-0.csv") as f:
    for row in csv.DictReader(f): eta_final = float(row["eta"])

plt.figure(figsize=(8, 5))
plt.plot(R0, T0, label="train (32 problems)", lw=1.5)
er = [r for r, e in zip(R0, E0) if e is not None]
ee = [e for e in E0 if e is not None]
if ee: plt.plot(er, ee, label="held-out eval (32 unseen)", lw=1.5)
plt.xlabel("server round"); plt.ylabel("cross-entropy loss")
plt.title("2-honest-worker gate training — train vs held-out (K=5000, Qwen-0.5B)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig("/kaggle/working/learn2/descent2.png", dpi=120)
print("saved descent2.png")

def ends(xs):
    xs = [x for x in xs if x is not None]
    return (xs[0], xs[-1], xs[-1]-xs[0]) if xs else (None, None, None)
t0v, t1v, td = ends(T0); e0, e1, ed = ends(E0)

# single-worker baseline (from the prior run, for the comparison line)
BASE = "held-out Δ = −0.0029 over 150 updates, η frozen 1e-3"
def f4(x): return f"{x:.4f}" if x is not None else "—"
def fd(x): return f"{x:+.4f}" if x is not None else "—"
md_txt = f"""# Phase 40 — 2-honest-worker learning run

- **Corpus:** 32 train / 32 held-out (disjoint, same as single-worker run).
- **Workers:** 2 honest, one Qwen-0.5B per GPU, K=5000 gates, deployed coordinator.
- **Server rounds:** {len(R0)}.  **η final:** {f4(eta_final)} (init 1e-3 — climbs if sym-AIMD fired).

| metric | start | final | Δ |
|---|---|---|---|
| train loss | {f4(t0v)} | {f4(t1v)} | {fd(td)} |
| held-out eval loss | {f4(e0)} | {f4(e1)} | {fd(ed)} |

**Single-worker baseline for comparison:** {BASE}.

**Read:** compare held-out Δ and final η against the baseline. A larger held-out drop and/or
η climbing above 1e-3 confirms the full-rate + adaptation hypothesis. Eval was never trained on,
so a monotone drop remains a generalisation signal.

![descent2](descent2.png)
"""
open("/kaggle/working/learn2/RESULT2.md", "w").write(md_txt)
print(md_txt)

## Cell 6 — outputs

In [ ]:
import os
print(os.listdir("/kaggle/working/learn2"))